# Module 17 Assignment: YOLOv11 Object Detection REST API, Testing & Dockerization

এই নোটবুকটিতে **Ostad AI Engineers - Module 17** এসাইনমেন্টের সম্পূর্ণ সমাধান ক্রমান্বয়ে দেওয়া হলো।

---

## 📌 ধাপ ১: এনভাইরনমেন্ট সেটআপ এবং প্রয়োজনীয় লাইব্রেরি ইনস্টলেশন (Setup Environment)
প্রথমেই Google Colab এ প্রয়োজনীয় প্যাকেজগুলো (Ultralytics, FastAPI, Uvicorn, Nest-Asyncio ইত্যাদি) ইনস্টল করে নেওয়া হচ্ছে।

In [ ]:
# ==============================================================================
# ধাপ ১: লাইব্রেরি ইনস্টল করা
# ==============================================================================
!pip install -q ultralytics fastapi uvicorn python-multipart requests nest_asyncio pillow opencv-python-headless
print('✅ সকল ডিপেন্ডেন্সি সফলভাবে ইনস্টল হয়েছে!')

## 📌 Task 1: Model Integration & Inference Pipeline (15 Marks)
YOLOv11 Pretrained / Custom Model Weights লোড করা এবং সিঙ্গেল ইমেজে অবজেক্ট ডিটেকশন চালনা করা।

In [ ]:
# ==============================================================================
# ধাপ ১.১: লাইব্রেরি ইমপোর্ট এবং ইনফারেন্স ফাংশন তৈরি
# ==============================================================================
import io                                        # বাইট স্ট্রিম প্রসেস করার জন্য
from PIL import Image                            # ইমেজ ওপেন ও প্রসেসের জন্য PIL
from ultralytics import YOLO                      # YOLOv11 অবজেক্ট ডিটেকশন মডেলের জন্য

# YOLOv11 Pretrained Nano Model লোড করা
model = YOLO('yolo11n.pt')                       # YOLOv11 মডেল অবজেক্ট তৈরি

def predict_yolo(image_bytes, conf_threshold=0.25):
    """ইমেজ বাইট গ্রহণ করে ডিটেকশন ফলাফল রিটার্ন করে"""
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB') # PIL ইমেজে কনভার্ট
    results = model.predict(source=img, conf=conf_threshold, verbose=False) # প্রেডিকশন চালানো
    
    predictions = []
    for box in results[0].boxes:
        cls_id = int(box.cls[0].item())
        predictions.append({
            'class_name': model.names[cls_id],  # ডিনোমিনেশন/ক্লাসের নাম
            'confidence': round(float(box.conf[0].item()), 4), # কনফিডেন্স স্কোর
            'bounding_box': [round(x, 2) for x in box.xyxy[0].tolist()] # [xmin, ymin, xmax, ymax]
        })
    return {'total_detections': len(predictions), 'predictions': predictions}

print('✅ YOLOv11 Inference Pipeline প্রস্তুত!')

## 📌 Task 2: REST API Development using FastAPI (25 Marks)
FastAPI দিয়ে `/predict` (POST) এন্ডপয়েন্ট তৈরি করা যেখানে ইমেজ ফাইল ফাইল আপলোড করে JSON ফলাফল পাওয়া যাবে।

In [ ]:
# ==============================================================================
# ধাপ ২.১: FastAPI অ্যাপ এবং এন্ডপয়েন্ট তৈরি
# ==============================================================================
from fastapi import FastAPI, File, UploadFile, HTTPException, status
from fastapi.responses import JSONResponse

app = FastAPI(title="YOLOv11 Detection API")     # FastAPI অ্যাপ ইন্সট্যান্স

@app.get("/")
def index():
    return {"status": "running", "message": "YOLOv11 API is Active"}

@app.post("/predict")
async def predict_endpoint(file: UploadFile = File(...)):
    if not file or not file.filename:
        raise HTTPException(status_code=400, detail="ফাইল পাওয়া যায়নি।")
    
    content = await file.read()
    if len(content) == 0:
        raise HTTPException(status_code=400, detail="খালি (0 bytes) ফাইল!")
        
    res = predict_yolo(content)
    return {
        "success": True,
        "filename": file.filename,
        "total_detections": res["total_detections"],
        "predictions": res["predictions"]
    }

print('✅ FastAPI REST API স্ক্রিপ্ট সম্পন্ন!')

## 📌 Task 3: API Testing & Validation (10 Marks)
Colab ব্যাকগ্রাউন্ডে সার্ভার চালু করা এবং ৫টি টেস্ট ইমেজের ওপর API হ্যান্ডলিং ও রেসপন্স যাচাই।

In [ ]:
# ==============================================================================
# ধাপ ৩.১: টেস্ট ইমেজের জন্য ৫টি ইমেজ তৈরি ও টেস্ট পরিচালনা
# ==============================================================================
import os, uvicorn, threading, time, requests
from PIL import Image, ImageDraw

# ব্যাকগ্রাউন্ডে Uvicorn সার্ভার চালু করার থ্রেড
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(2) # সার্ভার চালু হওয়া পর্যন্ত অপেক্ষা

# ৫টি টেস্ট ইমেজ তৈরি
os.makedirs("test_images", exist_ok=True)
for i in range(1, 6):
    img = Image.new("RGB", (300, 300), color=(50*i, 200-30*i, 100+20*i))
    draw = ImageDraw.Draw(img)
    draw.rectangle([50, 50, 200, 200], fill=(200, 50, 50))
    img.save(f"test_images/test{i}.jpg")

# ৫টি ইমেজে API রিকোয়েস্ট পাঠানো
print("--- API Testing Results ---")
for i in range(1, 6):
    file_path = f"test_images/test{i}.jpg"
    with open(file_path, "rb") as f:
        resp = requests.post("http://127.0.0.1:8000/predict", files={"file": (f"test{i}.jpg", f, "image/jpeg")})
        print(f"File: test{i}.jpg | Status: {resp.status_code} | Detections: {resp.json().get('total_detections')}")


## 📌 Task 4: Dockerization (30 Marks)
Docker ⚙️ `Dockerfile` এবং `requirements.txt` তৈরি।

In [ ]:
# ==============================================================================
# ধাপ ৪.১: Dockerfile এবং requirements.txt ফাইল তৈরি
# ==============================================================================
dockerfile_content = """FROM python:3.10-slim
WORKDIR /app
RUN apt-get update && apt-get install -y libgl1 libglib2.0-0 && rm -rf /var/lib/apt/lists/*
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app /app/app
COPY models /app/models
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""
with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)

reqs = "fastapi\nuvicorn\nultralytics\npillow\nopencv-python-headless\npython-multipart\nrequests\n"
with open("requirements.txt", "w") as f:
    f.write(reqs)

print('✅ Dockerfile এবং requirements.txt সফলভাবে তৈরি হয়েছে!')

## 📌 Task 5: Summary & Submission Checklist (20 Marks)
- [x] YOLOv11 Inference Pipeline
- [x] FastAPI REST API (`/predict` POST)
- [x] API Testing on 5 images
- [x] Dockerfile & Requirements
- [x] README Documentation